In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    BitsAndBytesConfig,
    Mistral3ForConditionalGeneration
)


model_id = "mistralai/Ministral-3-3B-Instruct-2512-BF16"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# 1. Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = "left" # Always keep padding "left" for inference

# 2. Load model in BF16 with automatic device placement
#    torch.bfloat16 matches the Hub weights. device_map="auto" sends layers 
#    to GPU if available, otherwise CPU.
model = Mistral3ForConditionalGeneration.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    quantization_config=bnb_config,
    device_map="auto"
)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/198k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/147k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/45.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

In [3]:
import json

JSON_SCHEMA = {
    "summary": "A concise, 1-2 sentence abstractive summary of the clinical scenario.",
    "clinical_reasoning": "A step-by-step logical breakdown of the diagnoses, treatments, or clinical decisions made in the text. Explain WHY certain relationships exist. Keep short and brief but to the point.",
    "relationships": [
        {
            "subject": "Source entity (e.g., Patient, Drug, Symptom)",
            "predicate": "Use STANDARD POSITIVE relationships (e.g., HAS_HISTORY, SHOWS_SYMPTOM, DIAGNOSED_WITH, PRESCRIBED). Do not use negated verbs like 'DENIES' or 'LACKS'.",
            "object": "Target entity",
            "polarity": "positive OR negative (Use 'negative' if the patient denies the history or lacks the symptom)",
            "certainty": "confirmed, suspected, OR hedged",
            "evidence": "The exact verbatim text snippet that proves this relationship."
        }
    ],
    "keywords": ["List", "of", "important", "clinical", "NER", "terms"]
}
SCHEMA_STRING = json.dumps(JSON_SCHEMA, indent=4)

SYSTEM_PROMPT = (
    "You are an expert clinical informatician. "
    f"Extract data strictly into this JSON schema:\n\n{SCHEMA_STRING}\n\n"
    "CRITICAL RULES:\n"
    "1. Use ONLY double quotes for all JSON keys and string values.\n"
    "2. Response MUST start with '{' and end with '}'.\n"
    "3. Output raw JSON only — no markdown, no code blocks.\n"
    "4. Provide values for ALL keys in the schema.\n"
    "5. Apostrophes in clinical terms (e.g., patient's) are allowed inside double-quoted strings.\n"
    "6. Extract at max 10 most clinically significant relationships only. "
    "Prioritize: diagnosis > treatment > symptoms > history.\n"
)

RAW_MEDICAL_TEXT = (
    "This case study describes a 48-year-old female patient who was diagnosed with T1DM at the age of 16.\n"
    "The patient remained under regular follow-up while conducting a labile glycemic control by first using insulin NPH twice a day and,"
    "subsequently, using a long- acting human insulin analog (from 1999 to present)\n"
    "There was no family history related to DM, systemic arterial hypertension or CAD.\n"
    "Smoking habits were absent during all her life.\n"
    "Her father died from stroke after the age of 70 years and her mother died from acute myocardial infarction when she was 75 years.\n"
    "Lipid profile and body mass index (BMI) and Blood pressure levels were normal, as were acute phase markers (C-reactive protein - CRP, homocysteine, apoprotein A and apoprotein B).\n"
    "Accordingly, this patient had a low risk for CAD.\n"
    "Proteinuria levels were within normality until the age of 40 years and after this urinary albuminuria excretion rate (UAER) was either within normality.\n"
    "At the age of 42 (26 years of disease evolution), she presented for the first time with clinical symptoms of acute coronary syndrome (ACS) with an acute myocardial infarction (AMI).\n"
    "At the age of 48, her last evaluation disclosed a predominantly systolic hypertension, and her myocardial scintigraphy showed an ischemia induced by effort, despite the optimized cardiovascular therapy.\n"
)

In [4]:
# 3. Prepare a chat-formatted prompt
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f"CONTEXT:\n\n{RAW_MEDICAL_TEXT}\n"},
]

# Mistal instruction models expect a specific chat template
prompt = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True
)

# 4. Tokenize and move to the same device as the model
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# 5. Generate
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=4096,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

# 6. Decode and print
response = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)

Both `max_new_tokens` (=4096) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [5]:
print(response)

```json
{
    "summary": "A 48-year-old female with Type 1 Diabetes Mellitus (T1DM) since age 16, long-term on long-acting insulin analog, demonstrates no family history of DM, hypertension, or CAD, but developed ACS with AMI at age 42 and persistent ischemia on stress testing at age 48 despite optimized therapy.",

    "clinical_reasoning":
    "1. **Diagnosis:** T1DM is confirmed with a long history of glycemic management via insulin (NPH and analog). Despite no family history of DM-related complications or CAD, ACS/AMI at age 42 indicates atherosclerotic risk not explained by conventional risk factors.
    2. **Key Observations:** Normal BMI, lipid profile, and BP initially but progression to hypertension at age 48 and inducible ischemia on myocardial scintigraphy suggests **late-onset atherosclerotic disease** likely driven by:
       - **Labile glycemic control** (chronic hyperglycemia → endothelial dysfunction, atherosclerosis).
       - **Long-term insulin analog use** (may alte